In [53]:
import pandas as pd

df = pd.read_csv('../data/cleaned_ds.csv')

In [54]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
import pandas as pd

level_1 = ['Low', 'Moderate', 'High']
level_2 = ['Poor', 'Average', 'Good']
yes_no = ['No', 'Yes']


categories = [level_1, level_2, yes_no, yes_no]
encoder = OrdinalEncoder(categories=categories)

col_to_encode = ['Physical_Activity', 'Diet_Quality', 'Part_Time_Job', 'Previous_Mental_Health_History']
df[col_to_encode] = encoder.fit_transform(df[col_to_encode]) 

df = pd.get_dummies(df, columns=['Gender'], drop_first=True, dtype=int)


X = df.drop(columns=['Mental_Health_Risk'])
y = df['Mental_Health_Risk']

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size= 0.2, 
    random_state=42, 
    stratify=y
    )

In [55]:
mapping = dict(zip(label_encoder.classes_, map(int,label_encoder.transform(label_encoder.classes_))))
print(mapping)

{'High': 0, 'Low': 1, 'Medium': 2}


In [56]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import ExtraTreeClassifier, DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegressionCV
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


models = [
    ExtraTreeClassifier(random_state=42, max_depth=5, min_samples_split=10),
    DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_split=10),
    RandomForestClassifier(random_state=42, max_depth=7, min_samples_split=10, n_estimators=100),
    HistGradientBoostingClassifier(random_state=42, learning_rate=0.1, max_iter=100, max_depth=5),
    LogisticRegressionCV(random_state=42, cv=5, max_iter=1000),
    XGBClassifier(n_estimato=100, random_state=42, learning_rate=0.1, max_depth=4),
    SVC(random_state=42, C=1, kernel='rbf', gamma='scale')
]

for model in models:

    model_name = type(model).__name__
    print(f'********{model_name}*********')

    pipline = Pipeline([
        ('sclare', StandardScaler()),
        ('model', model)
    ])

    pipline.fit(X_train, y_train)
    y_pred = pipline.predict(X_test)

    print(f'Accuracy : {accuracy_score(y_test, y_pred):4f}\n')
    print('confusion metrix :')
    print(confusion_matrix(y_test, y_pred))
    print('classification report : ')
    print(classification_report(y_test, y_pred))
    print('\n' + '='*50 + '\n')





********ExtraTreeClassifier*********
Accuracy : 0.502500

confusion metrix :
[[100  22  14]
 [ 36  77  19]
 [ 53  55  24]]
classification report : 
              precision    recall  f1-score   support

           0       0.53      0.74      0.62       136
           1       0.50      0.58      0.54       132
           2       0.42      0.18      0.25       132

    accuracy                           0.50       400
   macro avg       0.48      0.50      0.47       400
weighted avg       0.48      0.50      0.47       400



********DecisionTreeClassifier*********
Accuracy : 0.592500

confusion metrix :
[[ 82  15  39]
 [  5 109  18]
 [ 36  50  46]]
classification report : 
              precision    recall  f1-score   support

           0       0.67      0.60      0.63       136
           1       0.63      0.83      0.71       132
           2       0.45      0.35      0.39       132

    accuracy                           0.59       400
   macro avg       0.58      0.59      0.58   

In [57]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

pipline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

grid_params = {
        'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
        'model__l1_ratio': [0, 0.5, 1],
        # 'model__penalty': ['l1', 'l2', 'elasticnet'],
        'model__solver': ['saga', 'liblinear', 'lbfgs', 'newton-cg']
    }

grid_search = GridSearchCV(
    estimator=pipline,
    param_grid=grid_params,
    cv=5,
    scoring='accuracy',
    n_jobs=1
)

grid_search.fit(X_train, y_train)

print(f'Best params: {grid_search.best_params_}')
print(f'CV Accuracy: {grid_search.best_score_:.4f}')
print(f'Test Accuracy: {grid_search.score(X_test, y_test):.4f}')
# print(f'cv detailed results: {grid_search.cv_results_}')


Best params: {'model__C': 0.1, 'model__l1_ratio': 0, 'model__solver': 'saga'}
CV Accuracy: 0.7538
Test Accuracy: 0.7600


In [58]:
import joblib

joblib.dump(grid_search.best_estimator_ , '../model/mental_health_LR_model.joblib')

['../model/mental_health_LR_model.joblib']